# 12. Artificial Neural Network (ANN / MLP)
**Use case:** Insurance — Life insurance policy lapse prediction for retention outreach

When a life policy lapses, the carrier loses years of future premium and the family loses its protection, often without realising it. Insurers hold policy data such as policy age, premium changes, missed payments, autopay status, agent contacts and cash value. The retention team can only call a fraction of the book each month, so it needs a ranked list of who is most likely to lapse. Success means the top-ranked calls catch a large share of future lapses, probabilities are trustworthy for budgeting, and the model is documented for regulators.

> 💡 Think of a cricket selection panel: each selector (neuron) weighs a few stats differently and gives an opinion, a second panel weighs those opinions, and a chairman makes the final call. Training is the panel reviewing past mistakes and adjusting how much each selector trusts each stat.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: One neuron learns who will lapse
Train a single sigmoid neuron with hand-written forward pass and backpropagation on 8 policyholders.

| 5W + H | |
|---|---|
| **Who** | A retention analyst curious how a neural network actually learns. |
| **What** | Eight policyholders described by missed payments and premium increase, labelled lapsed or not. |
| **When** | Before using any library, to see the learning loop with nothing hidden. |
| **Where** | A plain Python script with only numpy. |
| **Why** | Every big network is millions of these neurons; understanding one makes the rest less magic. |
| **How** | Compute weighted sum, apply sigmoid, measure log-loss, compute gradients, update weights 300 times. |

**🧠 ML concepts:** Neuron as weighted sum plus bias, Sigmoid activation, Log-loss, Backpropagation for one neuron, Gradient descent and learning rate  
**🐍 Python concepts:** Defining a function with def, numpy matrix-vector multiply with @, Broadcasting a scalar bias, In-place update operators -=, zip and enumerate together, f-string alignment

In [ ]:
import numpy as np

# 8 life-insurance policyholders: [missed_payments_last_year, premium_increase_pct/10]
# 🐍 PY: each inner list is one customer; np.array makes it a 2-D table of floats
X = np.array([[0, 0.0], [0, 0.5], [1, 0.2], [0, 1.0],
              [2, 1.5], [3, 0.8], [2, 2.0], [4, 1.2]], dtype=float)
# 🐍 PY: 1 = the policy lapsed (customer stopped paying), 0 = still in force
y = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=float)

# 🧠 ML: one neuron = weighted sum of inputs + bias, squashed by sigmoid into a probability
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

np.random.seed(42)
# 🐍 PY: np.random.randn(2) gives two small random starting weights
w = np.random.randn(2) * 0.1
b = 0.0
lr = 0.5  # learning rate: how big each correction step is

# 🐍 PY: range(301) loops epochs 0..300; one epoch = one pass over all 8 customers
for epoch in range(301):
    # Forward pass: predict a lapse probability for every customer at once
    # 🐍 PY: X @ w is matrix-vector multiply; adding b broadcasts the bias to all 8 rows
    p = sigmoid(X @ w + b)
    # 🧠 ML: log-loss punishes confident wrong answers heavily
    loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
    # 🧠 ML: backprop for one neuron - the gradient of log-loss w.r.t. z is simply (p - y)
    error = p - y
    grad_w = X.T @ error / len(y)
    grad_b = error.mean()
    # 🧠 ML: gradient descent nudges weights against the gradient to reduce loss
    w -= lr * grad_w
    b -= lr * grad_b
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  loss={loss:.3f}  w={np.round(w, 2)}  b={b:.2f}")

print("\nCustomer  missed  prem_inc  lapsed  P(lapse)")
# 🐍 PY: zip walks X rows, labels and predictions in step; enumerate adds a row number
for i, (row, label, prob) in enumerate(zip(X, y, sigmoid(X @ w + b))):
    print(f"{i:>8}  {row[0]:>6.0f}  {row[1]*10:>7.0f}%  {label:>6.0f}  {prob:>8.2f}")

new_customer = np.array([1, 1.0])  # 1 missed payment, 10% premium increase
print(f"\nNew customer lapse probability: {sigmoid(new_customer @ w + b):.2f}")

**Try this next:**
- Set lr = 5.0 and see whether the loss jumps around instead of falling smoothly.
- Add a third feature column (on_autopay 0/1) and a third weight.
- Flip one label to make the data noisy and see the final loss stay higher.

## MEDIUM: sklearn MLP ranks 4,000 policies for outreach
Train a scaled two-hidden-layer MLPClassifier on 4,000 synthetic policies and turn scores into an outreach list.

| 5W + H | |
|---|---|
| **Who** | The retention analytics team supporting the outbound call centre. |
| **What** | 4,000 policies with 8 features and a lapse flag, modelled with an sklearn neural net. |
| **When** | Monthly, before the call centre plans its outreach. |
| **Where** | A Python notebook or scheduled job against the policy data mart. |
| **Why** | Calling the riskiest 10% should catch far more lapses than random calls. |
| **How** | Pipeline of StandardScaler and MLPClassifier with early stopping, stratified split, ROC-AUC and a top-decile lift check. |

**🧠 ML concepts:** Feature scaling for neural nets, Hidden layers and ReLU, Early stopping with patience, ROC-AUC vs accuracy on imbalanced data, Lift in the top decile  
**🐍 Python concepts:** Building a DataFrame from a dict, Vectorised boolean logic with | and comparisons, make_pipeline and indexing a step with [-1], DataFrame.assign and sort_values, groupby then mean

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
n = 4000

# --- 1. Synthetic life-insurance policy book ---
# 🐍 PY: a dict of column -> numpy array becomes a DataFrame in one step
df = pd.DataFrame({
    "policy_age_years": rng.integers(0, 25, n),
    "insured_age": rng.integers(22, 75, n),
    "annual_premium_usd": rng.lognormal(7.3, 0.5, n).round(0),
    "premium_increase_pct": rng.choice([0, 0, 0, 5, 10, 20], n),
    "missed_payments_12m": rng.poisson(0.5, n),
    "on_autopay": rng.integers(0, 2, n),
    "agent_contacts_12m": rng.poisson(1.5, n),
    "cash_value_usd": rng.lognormal(8, 1.0, n).round(0),
})

# 🧠 ML: the true lapse rule is NON-linear (interactions, a U-shape in policy age) - good for a neural net
z = (-2.2 + 0.9 * df.missed_payments_12m + 0.06 * df.premium_increase_pct
     - 1.1 * df.on_autopay - 0.25 * df.agent_contacts_12m
     + 0.8 * ((df.policy_age_years < 3) | (df.policy_age_years > 20))
     + 0.04 * df.premium_increase_pct * (1 - df.on_autopay)
     - 0.3 * np.log(df.cash_value_usd / 3000))
# 🐍 PY: comparing a random uniform number to a probability draws a 0/1 outcome per row
df["lapsed"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
print(df.head(3).to_string(index=False))
print(f"\nPolicies: {n}, lapse rate: {df.lapsed.mean():.1%}")

# --- 2. Train/test split ---
X = df.drop(columns="lapsed")
y = df["lapsed"]
# 🧠 ML: stratify=y keeps the same lapse rate in train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# --- 3. Neural network with scaling ---
# 🧠 ML: neural nets need scaled inputs - premiums in thousands would swamp 0/1 autopay flags
# 🧠 ML: hidden_layer_sizes=(16, 8) = two hidden layers with 16 and 8 neurons
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8), activation="relu", alpha=1e-3,
                  max_iter=500, early_stopping=True, n_iter_no_change=25,
                  random_state=42),
)
model.fit(X_train, y_train)
# 🐍 PY: [-1] grabs the last step of the pipeline; n_iter_ = epochs actually run
print(f"Training stopped after {model[-1].n_iter_} epochs")

# --- 4. Evaluate ---
# 🧠 ML: predict_proba gives P(lapse); column 1 is the positive class
proba = model.predict_proba(X_test)[:, 1]
print(f"\nTest ROC-AUC: {roc_auc_score(y_test, proba):.3f}")
print(classification_report(y_test, proba >= 0.5, target_names=["in_force", "lapsed"], digits=3))

# --- 5. Business interpretation: who should the retention team call? ---
# 🐍 PY: assign adds a new column without modifying X_test in place
scored = X_test.assign(p_lapse=proba, lapsed=y_test.values)
top = scored.sort_values("p_lapse", ascending=False).head(len(scored) // 10)
print(f"Top 10% riskiest policies: actual lapse rate {top.lapsed.mean():.1%} "
      f"vs book average {scored.lapsed.mean():.1%}")
print(f"Calling them reaches {top.lapsed.sum() / scored.lapsed.sum():.0%} of all lapses "
      f"with {len(top)} calls.")
# 🐍 PY: groupby + mean shows the average predicted risk per autopay status
print("\nAverage predicted lapse risk by autopay:")
print(scored.groupby("on_autopay")["p_lapse"].mean().round(3).to_string())

**Try this next:**
- Remove StandardScaler from the pipeline and compare the ROC-AUC.
- Change hidden_layer_sizes to (4,) and then (64, 32) - does bigger help?
- Set n_iter_no_change back to 10 (the default) and watch training stop too early.

## HARD: PyTorch lapse model with weights, calibration and budget
Build a PyTorch MLP with class weights and early stopping, benchmark against logistic regression, explain it, calibrate it and pick an outreach volume.

| 5W + H | |
|---|---|
| **Who** | Retention data scientists, with actuarial and model-governance reviewers. |
| **What** | A production-style lapse scorer with a challenger baseline, explanations, calibrated probabilities and a saved model. |
| **When** | Retrained quarterly and scored monthly; calibration checked each quarter. |
| **Where** | Training in a Python job; the saved weights and scaler served by the lapse model API. |
| **Why** | Insurers must justify models to regulators, and finance needs real probabilities to budget the retention programme. |
| **How** | Train/val/test split, train-only scaling, BCEWithLogitsLoss with pos_weight, mini-batch Adam, early stopping, permutation importance, decile calibration, odds correction and a cost-benefit cut-off. |

**🧠 ML concepts:** Class imbalance with pos_weight, Mini-batch training and dropout, Early stopping on validation loss, Challenger baseline comparison, Permutation importance, Probability calibration after re-weighting  
**🐍 Python concepts:** Subclassing nn.Module with __init__ and forward, Dict comprehension to copy model weights, torch.no_grad context manager, Functions returning pandas Series, List of tuples in a for loop, String repetition for ASCII bars

In [ ]:
"""Life-insurance lapse model: PyTorch MLP vs logistic regression, class weights, early stopping,
permutation importance, calibration and an outreach-budget threshold."""
import os
import tempfile
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)
FEATURES = ["policy_age_years", "insured_age", "annual_premium_usd", "premium_increase_pct",
            "missed_payments_12m", "on_autopay", "agent_contacts_12m", "cash_value_usd"]


def make_policies(n=5000):
    df = pd.DataFrame({
        "policy_age_years": rng.integers(0, 25, n), "insured_age": rng.integers(22, 75, n),
        "annual_premium_usd": rng.lognormal(7.3, 0.5, n), "premium_increase_pct": rng.choice([0, 0, 0, 5, 10, 20], n),
        "missed_payments_12m": rng.poisson(0.5, n), "on_autopay": rng.integers(0, 2, n),
        "agent_contacts_12m": rng.poisson(1.5, n), "cash_value_usd": rng.lognormal(8, 1.0, n)})
    # 🧠 ML: interactions and a U-shape - patterns a straight-line model cannot fully capture
    z = (-2.8 + 0.9 * df.missed_payments_12m + 0.05 * df.premium_increase_pct - 1.1 * df.on_autopay
         - 0.25 * df.agent_contacts_12m + 1.0 * ((df.policy_age_years < 3) | (df.policy_age_years > 20))
         + 0.06 * df.premium_increase_pct * (1 - df.on_autopay) - 0.3 * np.log(df.cash_value_usd / 3000))
    df["lapsed"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    return df


class LapseMLP(nn.Module):
    """Two hidden layers with ReLU and dropout; outputs one logit per policy."""
    def __init__(self, n_in, h1=32, h2=16, p_drop=0.1):
        super().__init__()
        # 🧠 ML: Dropout randomly switches off neurons in training, a cheap guard against overfitting
        self.net = nn.Sequential(nn.Linear(n_in, h1), nn.ReLU(), nn.Dropout(p_drop),
                                 nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(Xtr, ytr, Xval, yval, epochs=200, patience=15, lr=3e-3):
    model = LapseMLP(Xtr.shape[1])
    # 🧠 ML: pos_weight up-weights the rarer lapsed class so the net does not just predict 'stays'
    pos_weight = torch.tensor((ytr == 0).sum() / (ytr == 1).sum(), dtype=torch.float32)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)
    Xv, yv = torch.tensor(Xval, dtype=torch.float32), torch.tensor(yval, dtype=torch.float32)
    best, best_state, wait = np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        # 🧠 ML: mini-batches of 128 - each step sees a random slice, which speeds and regularises training
        for idx in torch.randperm(len(Xt)).split(128):
            opt.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv), yv).item()
        # 🧠 ML: early stopping - keep the weights from the epoch with the best validation loss
        if val_loss < best - 1e-4:
            best, best_state, wait = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    print(f"MLP early-stopped at epoch {epoch + 1}, best val loss {best:.4f}")
    return model


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X, dtype=torch.float32))).numpy()


def permutation_importance(model, X, y, n_repeats=5):
    """Drop in ROC-AUC when one column is shuffled = how much the model relies on it."""
    base = roc_auc_score(y, predict(model, X))
    out = {}
    for j, name in enumerate(FEATURES):
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[:, j] = rng.permutation(Xp[:, j])
            drops.append(base - roc_auc_score(y, predict(model, Xp)))
        out[name] = np.mean(drops)
    return pd.Series(out).sort_values(ascending=False)


def calibration_table(y, p, bins=5):
    # 🧠 ML: calibration = do policies scored 30% actually lapse about 30% of the time?
    q = pd.qcut(p, bins, labels=False, duplicates="drop")
    return pd.DataFrame({"pred": p, "actual": y, "bin": q}).groupby("bin").mean().round(3)


if __name__ == "__main__":
    df = make_policies()
    df["annual_premium_usd"] = np.log(df.annual_premium_usd)
    df["cash_value_usd"] = np.log(df.cash_value_usd)
    X, y = df[FEATURES].values, df.lapsed.values
    print(f"Policies {len(df)}, lapse rate {y.mean():.1%}")
    Xtmp, Xte, ytmp, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    Xtr, Xval, ytr, yval = train_test_split(Xtmp, ytmp, test_size=0.2, stratify=ytmp, random_state=42)
    # 🧠 ML: fit the scaler on TRAIN only, then apply to val/test - avoids leaking test statistics
    scaler = StandardScaler().fit(Xtr)
    Xtr, Xval, Xte = scaler.transform(Xtr), scaler.transform(Xval), scaler.transform(Xte)

    mlp = train_mlp(Xtr, ytr, Xval, yval)
    p_mlp = predict(mlp, Xte)
    # 🧠 ML: baseline - a well-understood logistic regression with the same class weighting
    logit = LogisticRegression(class_weight="balanced", max_iter=1000).fit(Xtr, ytr)
    p_lr = logit.predict_proba(Xte)[:, 1]

    print("\nModel        ROC-AUC  PR-AUC  Brier")
    for name, p in [("Logistic", p_lr), ("MLP", p_mlp)]:
        print(f"{name:<12} {roc_auc_score(yte, p):.3f}    {average_precision_score(yte, p):.3f}   "
              f"{brier_score_loss(yte, p):.3f}")

    print("\nPermutation importance (ROC-AUC drop) for the MLP:")
    for feat, val in permutation_importance(mlp, Xte, yte).items():
        print(f"  {feat:<22} {val:+.4f} {'#' * int(max(val, 0) * 200)}")

    print("\nCalibration of MLP (class weights inflate probabilities):")
    print(calibration_table(yte, p_mlp).to_string())
    # 🧠 ML: simple fix - rescale odds by the class weight so scores become real lapse probabilities
    w = (ytr == 0).sum() / (ytr == 1).sum()
    odds = p_mlp / (1 - p_mlp) / w
    p_cal = odds / (1 + odds)
    print(f"Mean predicted after odds correction {p_cal.mean():.3f} vs actual {yte.mean():.3f}; "
          f"Brier {brier_score_loss(yte, p_cal):.3f}")

    # 🧠 ML: business threshold - retention team can call 15% of the book; value saved per lapse
    budget = int(0.15 * len(yte))
    call_cost, saved_value, save_rate = 40, 1800, 0.25
    top = np.argsort(-p_cal)[:budget]
    caught = yte[top].sum()
    net = caught * save_rate * saved_value - budget * call_cost
    print(f"\nCall top {budget} policies: reach {caught}/{yte.sum()} lapses "
          f"({caught / yte.sum():.0%}); expected net value ${net:,.0f}")

    path = os.path.join(tempfile.mkdtemp(), "lapse_mlp.pt")
    torch.save({"state_dict": mlp.state_dict(), "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_}, path)
    print(f"Saved model -> {os.path.basename(path)}")
    print("Governance: NAIC AI Model Bulletin - document data, test for unfair discrimination,")
    print("keep the logistic baseline as a challenger, and monitor calibration quarterly.")

**Try this next:**
- Remove pos_weight from the loss and compare PR-AUC and the calibration table.
- Change the call budget from 15% to 5% and 30% and find the best net value.
- Remove the interaction term from make_policies and see whether the MLP still beats logistic regression.